# Auditing analytical SQL on real data

**Question:** What is the mean body mass among recorded measurements in the Palmer penguins snapshot, and which plausible analyses misstate that quantity?

This notebook is an independent methods demonstration with five constructed flawed-analysis cases. It is not a model transcript, employer assessment, or causal study. Source data: Horst, Hill & Gorman (2020), palmerpenguins, CC0-1.0; original study Gorman et al. (2014), doi:10.1371/journal.pone.0090081.

In [1]:
import json
import html
import math
from IPython.display import HTML, display
from sql_analysis_audit.audit import (DATA, SQL, verify_snapshot, provenance, load_rows,
    create_database, query, build_report)

def table(rows):
    keys=list(rows[0]) if rows else []
    header=''.join('<th>'+html.escape(str(k))+'</th>' for k in keys)
    body=''.join('<tr>'+''.join('<td>'+html.escape(str(row[k]))+'</td>' for k in keys)+'</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))

print('Verified SHA-256:', verify_snapshot())
print('Upstream commit:', provenance()['upstream_commit'])
connection=create_database(load_rows())
report=build_report()

Verified SHA-256: f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93
Upstream commit: 8957207b78d6ccd1b4654a9dd9c9041b657478ab


## 1. Reconcile the sample before interpreting a mean

`COUNT(*)` counts source rows, while `COUNT(body_mass_g)` excludes missing masses. All `NA` fields remain SQL NULL. File-ordinal IDs are technical row keys, not biological subject identifiers.

In [2]:
print((SQL/'profile.sql').read_text())
table(query(connection,'profile'))
assert report['profile']['n']==344
assert report['profile']['mass_observed']==342
assert report['profile']['complete_cases']==333

SELECT COUNT(*) AS n,
       COUNT(body_mass_g) AS mass_observed,
       COUNT(*) - COUNT(body_mass_g) AS mass_missing,
       COUNT(sex) AS sex_observed,
       SUM(CASE WHEN bill_length_mm IS NOT NULL AND bill_depth_mm IS NOT NULL
                 AND flipper_length_mm IS NOT NULL AND body_mass_g IS NOT NULL
                 AND sex IS NOT NULL THEN 1 ELSE 0 END) AS complete_cases,
       AVG(body_mass_g) AS mean_mass_g,
       MIN(year) AS first_year, MAX(year) AS last_year
FROM observations;



n,mass_observed,mass_missing,sex_observed,complete_cases,mean_mass_g,first_year,last_year
344,342,2,333,333,4201.754385964912,2007,2009


## 2. Species aggregation and window functions

The window denominator is the total number of observations; `DENSE_RANK` ranks the descriptive group means. These ranks do not measure a treatment effect or infer anything about unobserved populations.

In [3]:
print((SQL/'species.sql').read_text())
table(query(connection,'species'))
assert math.isclose(sum(r['observation_share'] for r in report['species']),1.0)

WITH grouped AS (
  SELECT species, COUNT(*) AS n, COUNT(body_mass_g) AS mass_observed,
         AVG(body_mass_g) AS mean_mass_g
  FROM observations GROUP BY species
)
SELECT species, n, mass_observed, mean_mass_g,
       1.0 * n / SUM(n) OVER () AS observation_share,
       DENSE_RANK() OVER (ORDER BY mean_mass_g DESC) AS mass_rank
FROM grouped ORDER BY species;



species,n,mass_observed,mean_mass_g,observation_share,mass_rank
Adelie,152,151,3700.662251655629,0.4418604651162791,3
Chinstrap,68,68,3733.0882352941176,0.19767441860465115,2
Gentoo,124,123,5076.016260162602,0.36046511627906974,1


## 3. Yearly sample changes with LAG

The difference from the preceding year's sample mean is not within-animal growth. The simplified table does not retain identities sufficient for that conclusion.

In [4]:
print((SQL/'yearly.sql').read_text())
table(query(connection,'yearly'))
assert len(report['yearly'])==9

WITH grouped AS (
  SELECT species, year, COUNT(*) AS n, COUNT(body_mass_g) AS mass_observed,
         AVG(body_mass_g) AS mean_mass_g
  FROM observations GROUP BY species, year
), lagged AS (
  SELECT *, LAG(mean_mass_g) OVER (PARTITION BY species ORDER BY year)
         AS prior_sample_mean_g FROM grouped
)
SELECT *, mean_mass_g - prior_sample_mean_g AS sample_mean_change_g
FROM lagged ORDER BY species, year;



species,year,n,mass_observed,mean_mass_g,prior_sample_mean_g,sample_mean_change_g
Adelie,2007,50,49,3696.4285714285716,None,None
Adelie,2008,50,50,3742.0,3696.4285714285716,45.57142857142844
Adelie,2009,52,52,3664.903846153846,3742.0,-77.09615384615381
Chinstrap,2007,26,26,3694.230769230769,None,None
Chinstrap,2008,18,18,3800.0,3694.230769230769,105.76923076923094
Chinstrap,2009,24,24,3725.0,3800.0,-75.0
Gentoo,2007,34,34,5070.588235294118,None,None
Gentoo,2008,46,46,5019.565217391304,5070.588235294118,-51.02301790281399
Gentoo,2009,44,43,5140.697674418605,5019.565217391304,121.13245702730092


## 4. Averages of averages need an explicit estimand

Equal-species weighting is legitimate for a stated equal-species estimand. It does not recover the pooled observation mean. Weighting must use nonmissing mass counts, not total subgroup sizes.

In [5]:
print((SQL/'weighted_mean.sql').read_text())
table(query(connection,'weighted_mean'))
assert math.isclose(report['weighted_means']['correct_pooled_mean_g'],report['profile']['mean_mass_g'])

WITH grouped AS (
  SELECT species, COUNT(body_mass_g) AS observed,
         AVG(body_mass_g) AS mean_mass_g
  FROM observations GROUP BY species
)
SELECT SUM(mean_mass_g * observed) / SUM(observed) AS correct_pooled_mean_g,
       AVG(mean_mass_g) AS equal_species_mean_g
FROM grouped;



correct_pooled_mean_g,equal_species_mean_g
4201.754385964912,4169.92224903745


## 5. Reproduce four computational review failures

The duplicate-tag relation is deliberately synthetic and used only to reproduce a many-to-many join mistake. It is not presented as real ecological metadata. Every candidate below is an authored negative control.

In [6]:
for case in report['review_cases'][:4]:
    print(json.dumps(case,indent=2))
assert report['review_cases'][2]['candidate_rows']==496
assert report['review_cases'][2]['correct_rows']==344

{
  "id": "null-denominator",
  "verdict": "reject",
  "claim": "SUM(body_mass_g) / COUNT(*) is the observed-mass mean.",
  "candidate_mean_g": 4177.325581395349,
  "correct_mean_g": 4201.754385964912,
  "explanation": "COUNT(*) includes missing masses while SUM ignores them. Use AVG(body_mass_g), or divide by COUNT(body_mass_g). The estimand is the mean among measured masses, not an imputed population mean."
}
{
  "id": "unweighted-subgroup-means",
  "verdict": "reject",
  "claim": "The equally weighted mean of species means is the pooled observation mean.",
  "candidate_mean_g": 4169.92224903745,
  "correct_mean_g": 4201.754385964912,
  "explanation": "Equal-species weighting answers a different question. Recover the observation-weighted mean using each species nonmissing mass count, not its total row count."
}
{
  "id": "join-fanout",
  "verdict": "reject",
  "claim": "Joining species tags preserves the number of observations.",
  "candidate_rows": 496,
  "correct_rows": 344,
  "gua

## 6. Descriptive association is not causation

The species-centered calculation subtracts each species mean from both variables and correlates the residuals. It is not the average of species-specific correlations. Neither that calculation nor the pooled correlation controls every relevant sampling or causal mechanism.

In [7]:
print((SQL/'association.sql').read_text())
table(report['association'])
print(json.dumps(report['review_cases'][4],indent=2))

WITH pairs AS (
  SELECT species, flipper_length_mm AS x, body_mass_g AS y
  FROM observations
  WHERE flipper_length_mm IS NOT NULL AND body_mass_g IS NOT NULL
), centered AS (
  SELECT x - AVG(x) OVER () AS dx, y - AVG(y) OVER () AS dy,
         x - AVG(x) OVER (PARTITION BY species) AS sx,
         y - AVG(y) OVER (PARTITION BY species) AS sy
  FROM pairs
)
SELECT 'pooled' AS scope, COUNT(*) AS n,
       SUM(dx*dx) AS ss_x, SUM(dy*dy) AS ss_y, SUM(dx*dy) AS cross_product
FROM centered
UNION ALL
SELECT 'species_centered', COUNT(*), SUM(sx*sx), SUM(sy*sy), SUM(sx*sy)
FROM centered ORDER BY scope;



scope,n,ss_x,ss_y,cross_product,pearson_r,slope_g_per_mm
pooled,342,67426.54093567251,219307697.36842105,3350125.8771929825,0.8712017673060114,49.68556640610009
species_centered,342,14953.257036621788,72443483.21290235,608678.320601379,0.5848173822637013,40.70540077728046


{
  "id": "causal-overclaim",
  "verdict": "not-supported",
  "claim": "The pooled mass/flipper association establishes a causal effect applicable to each species.",
  "pooled_r": 0.8712017673060114,
  "species_centered_r": 0.5848173822637013,
  "explanation": "Pooled and within-species centered associations answer different descriptive questions. Neither identifies causation. Species adjustment alone does not resolve sex, year, sampling, or other confounding."
}


## 7. Audit conclusion

Use the 342 recorded masses for the stated observed-mass mean. Preserve the missingness report; disclose any alternative sample or weighting choice. Reject the deliberately duplicated species dimension before joining. Report the pooled and species-centered associations only as descriptive quantities.

We do not attach p-values, confidence intervals, causal claims, predictive scores, or population claims to this small historical sample. SQL results are separately checked against independent Python calculations in the test suite.

In [8]:
assert report['join_audit']['fact_rows']==report['join_audit']['joined_rows']==344
assert len(report['review_cases'])==5
assert build_report()==report
print('Runtime:', json.dumps(report['runtime'],sort_keys=True))
print('SQL fingerprints:', json.dumps(report['sql_sha256'],indent=2,sort_keys=True))
print('All notebook assertions passed.')
connection.close()

Runtime: {"python": "3.13.3", "sqlite": "3.49.1"}
SQL fingerprints: {
  "association.sql": "ed6822cc997b70657716511aff890e1a9d4bf669832222dd40034a267381e401",
  "profile.sql": "a7bcc7bed036132167d6362e1c4b619c7654147e63facdb2f735215306aba845",
  "schema.sql": "1bd19609a2c44132139450386541c9995d9a91a121f8111726fedcf3c702549c",
  "species.sql": "d2e24d1f7ff997b8f91270631e3a014568f09121656f5637fcb2ed5c36ffe443",
  "weighted_mean.sql": "2183c703ad9d78b5a9973836ed23ff8176c0816cf470ad59a68269362a326b8c",
  "yearly.sql": "3c3a153a90d496cc65e789278c7c9e07a86f48d8b72e42ccbe57076058b4bf6e"
}
All notebook assertions passed.
